In [55]:
from quantmgmt.data import YahooFinanceData
from quantmgmt.optimizers.black_litterman import black_litterman
from quantmgmt.optimizers.mean_variance import max_sharpe
import numpy as np
import pandas as pd

## Parte A: Black-Litterman

### 1. Pesos de equilibrio

In [56]:
# Investment Universe – ETFs

etfs = {
    # Equities
    "SPY": "SPDR S&P 500 ETF Trust",
    "VGK": "Vanguard FTSE Europe ETF",
    "VWO": "Vanguard FTSE Emerging Markets ETF",
    "EWJ": "iShares MSCI Japan ETF",

    # Fixed Income
    "AGG": "iShares Core U.S. Aggregate Bond ETF",
    "TLT": "iShares 20+ Year Treasury Bond ETF",
    "IEF": "iShares 7-10 Year Treasury Bond ETF",
    "LQD": "iShares iBoxx $ Investment Grade Corporate Bond ETF",
    "TIP": "iShares TIPS Bond ETF",

    # Commodities
    "GLD": "SPDR Gold Shares",
    "DBC": "Invesco DB Commodity Index Tracking Fund"
}

# Asset allocation
market_weights = {
    # Equities: 60%
    "SPY": 0.15,
    "VGK": 0.15,
    "VWO": 0.15,
    "EWJ": 0.15,

    # Fixed Income: 30%
    "AGG": 0.06,
    "TLT": 0.06,
    "IEF": 0.06,
    "LQD": 0.06,
    "TIP": 0.06,

    # Commodities: 10%
    "GLD": 0.05,
    "DBC": 0.05
}

tickers = list(etfs.keys())
names = list (etfs.values())

print("INVESTMENT UNIVERSE: Strategic Benchmark Portfolio")
print("=" * 100)

for ticker, name in etfs.items():
    print(f"{ticker:<6} | {name:<65} | Weight: {market_weights[ticker]:>6.1%}")

print("=" * 100)
print(f"Number of assets: {len(tickers)}")
print(f"Total market weight: {sum(market_weights.values()):.1%}")

INVESTMENT UNIVERSE: Strategic Benchmark Portfolio
SPY    | SPDR S&P 500 ETF Trust                                            | Weight:  15.0%
VGK    | Vanguard FTSE Europe ETF                                          | Weight:  15.0%
VWO    | Vanguard FTSE Emerging Markets ETF                                | Weight:  15.0%
EWJ    | iShares MSCI Japan ETF                                            | Weight:  15.0%
AGG    | iShares Core U.S. Aggregate Bond ETF                              | Weight:   6.0%
TLT    | iShares 20+ Year Treasury Bond ETF                                | Weight:   6.0%
IEF    | iShares 7-10 Year Treasury Bond ETF                               | Weight:   6.0%
LQD    | iShares iBoxx $ Investment Grade Corporate Bond ETF               | Weight:   6.0%
TIP    | iShares TIPS Bond ETF                                             | Weight:   6.0%
GLD    | SPDR Gold Shares                                                  | Weight:   5.0%
DBC    | Invesco DB Commodity

In [57]:
TICKERS = [
    "SPY", "VGK", "VWO", "EWJ",
    "AGG", "TLT", "IEF", "LQD", "TIP",
    "GLD", "DBC"]
START, END = "2014-01-01", "2025-12-31"

loader = YahooFinanceData()
assets = loader.get_prices(TICKERS, START, END)[TICKERS]

print(f"{assets.index[0]:%Y-%m-%d} -> {assets.index[-1]:%Y-%m-%d}: {len(assets)} sesiones")

2014-01-02 -> 2025-12-30: 3017 sesiones


### 2. Optimización inversa

In [58]:
returns = assets.pct_change().dropna()
w_mkt = [w for w in market_weights.values()]
sigma = returns.cov() * 252
market_returns = returns @ np.array(w_mkt)
delta = (market_returns.mean()*252)/(market_returns.var()*252)


In [59]:
pi_equilibrium = delta*sigma@w_mkt
pi_equilibrium

SPY    0.101534
VGK    0.112429
VWO    0.113578
EWJ    0.097778
AGG    0.011160
TLT    0.004000
IEF    0.003425
LQD    0.025052
TIP    0.010051
GLD    0.026654
DBC    0.052166
dtype: float64

### 3. Visiones

Para este apartado, hemos analizado por encima el entorno macroeconómico. La idea es la de poder dar una opinión fundada sobre las views del ejercicio, no necesariamente acertar o predecir movimientos en los activos.

- **Crecimiento global:** La economía mundial mantiene un crecimiento moderado, aunque desigual entre regiones. El FMI estima un crecimiento del 3,1% en 2026.

- **Estados Unidos:** La economía muestra resiliencia, con un crecimiento relativamente sólido, pero la inflación sigue por encima del objetivo del 2% de la Reserva Federal.

- **Política monetaria:** La persistencia de las presiones inflacionistas limita el margen de los bancos centrales para reducir los tipos de interés, generando incertidumbre sobre su evolución.

- **Europa:** Presenta un crecimiento económico débil, condicionado por los elevados costes energéticos y la incertidumbre geopolítica.

- **Mercados emergentes:** Mantienen perspectivas de crecimiento relativamente favorables, aunque con importantes diferencias entre países y exposición a riesgos comerciales y cambiarios.

- **Japón:** Está experimentando mayores presiones inflacionistas y salariales, en un contexto de normalización económica y monetaria.

- **Materias primas:** Las tensiones geopolíticas, especialmente en Oriente Medio, incrementan la incertidumbre sobre los precios energéticos y pueden alimentar nuevas presiones inflacionistas.

- **Incertidumbre global:** Los conflictos geopolíticos, las tensiones comerciales y la persistencia de la inflación constituyen los principales riesgos para los mercados financieros, favoreciendo potencialmente la demanda de activos defensivos como el oro.

Con estas ideas en la mesa, podemos proponer algunas views:
1. Europa se encuentra últimamente algo lagging en cuanto a crecimiento y parece que se acercará a un escenario de estanflación más acuciantemente que Estados Unidos. La rentabilidad de los últimos 20 años del Vanguard FTSE Europe ETF ha sido un 3,70%. La del SPDR S&P 500 ETF Trust ha sido un 7.80% anual. Creemos que puede ampliarse este spread alrededor de un 1% más aproximadamente. Por lo tanto:

E[SPY] - E[VGK] = 5.00%

2. El oro puede continuar viéndose como activo refugio si las políticas de los bancos centrales de subidas de tipos no consiguen controlar la inflación. La rentabilidad que puede estar dando este año, será parecida a la histórica, digamos un 3.5%. La del oro, quizás algo superior de lo habitual, digamos un 8%. Entonces:

E[GLD] - E[TLT] = 4.50%

3. Por último es posible que los conflictos que están actualmente moviendo los mercados de commodities cesen o al menos se calmen basten durante el próximo año, por lo que esperamos una caída relativa de los índices de commodities con respecto al oro de:

E[DBC] - E[GLD] = -8.00%

In [60]:
P = np.array([
    [1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0,  0, 0, 0, 0, -1, 0, 0, 0, 1, 0],
    [0,  0, 0, 0, 0,  0, 0, 0, 0, -1, 1]
])
Q = np.array([
    0.05,
    0.045,
    -0.08
])

tau = 0.05

In [61]:
tau = 0.05

In [62]:
varianza_views = P@(tau * sigma)@P.transpose()

In [63]:
varianza_views

,0,1,2
0,0.000607,-0.000105,0.000062
1,-0.000105,0.001575,-0.000245
2,0.000062,-0.000245,0.001892


In [64]:
returns_bl = black_litterman(pi = pi_equilibrium, sigma = sigma, P = P, Q = Q, tau = tau)

In [65]:
w_bl = max_sharpe(returns_bl[0], sigma)

In [66]:
w_bl.sum()

np.float64(1.0)

In [67]:
df_comparación_pesos = pd.DataFrame()
df_comparación_pesos["Tickers"] = TICKERS
df_comparación_pesos["Nombres"] = names
df_comparación_pesos["Pesos Cartera"] = w_mkt
df_comparación_pesos["Peso Black Litterman"] = np.round(w_bl, 2)

df_comparación_pesos

,Tickers,Nombres,Pesos Cartera,Peso Black Litterman
0,SPY,SPDR S&P 500 ETF Trust,0.15,0.42
1,VGK,Vanguard FTSE Europe ETF,0.15,0.00
2,VWO,Vanguard FTSE Emerging Markets ETF,0.15,0.04
3,EWJ,iShares MSCI Japan ETF,0.15,0.09
4,AGG,iShares Core U.S. Aggregate Bond ETF,0.06,0.00
5,TLT,iShares 20+ Year Treasury Bond ETF,0.06,0.07
6,IEF,iShares 7-10 Year Treasury Bond ETF,0.06,0.14
7,LQD,iShares iBoxx $ Investment Grade Corporate Bon...,0.06,0.00
8,TIP,iShares TIPS Bond ETF,0.06,0.00
9,GLD,SPDR Gold Shares,0.05,0.24
